In [62]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import torch
import gc
from IPython.display import display

ds_name = 'real_NCT00981058_inject'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
proj_name = 'SurvSurfBenchmark_NCT00981058_inject'
g_resol = 1
t_resol = 7
g_max = 5
split='test'

In [63]:
DEVICE = 'cpu'
if DEVICE == 'gpu':
    assert torch.cuda.is_available()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = 'cpu'

## Import and instantiate model

In [64]:
from dataset_NCT00981058 import DataModuleNCT00981058, DatasetNCT00981058

In [65]:
ds = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split=split, 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False,
)
df_obs = ds._get_df_Xy_trans_obs()
df_true_prob = ds._get_df_Xy_true_prob()
df_true_prob.loc[
    df_true_prob[ds.colname_g] == 6,:
].head()

,subject,duration,g_max_by_time,event_observed,weight,feat__AGE,feat__BLOOD AND LYMPHATIC SYSTEM DISORDERS,feat__BMIBL,feat__BSABL,feat__Body Surface Area (m^2),...,feat__SMKB_Smoker (all others),feat__SOCIAL CIRCUMSTANCES,feat__SURGICAL AND MEDICAL PROCEDURES,feat__Systolic Blood Pressure (mmHg),feat__Temperature (°C),feat__VASCULAR DISORDERS,"feat__WBCCAT1_> 11,000 µl (11 x 10^9/L)",feat__Weight (kg),feat__traj_clust,is_t_trans


In [66]:
df_true_prob['g_max_by_time'].value_counts()

g_max_by_time
1    2400
2    2400
3    2400
4    2400
5    2400
Name: count, dtype: int64

In [67]:

def pred_from_SurvSurf(run_id, proj_name, split='val'):
    import wandb
    from model_factory_survsurf import LitModelSurvSurf
    torch.set_float32_matmul_precision('medium')

    api = wandb.Api()
    # artifact = api.artifact(f'yichenchen-wings/{proj_name}/model-{run_id}:v0', type='model')
    # checkpoint_dir = artifact.download()
    checkpoint_dir = f'/home/yc366/repos/survsurf_benchmark/runtime_results/{proj_name}/{run_id}/checkpoints/'
    os.listdir(checkpoint_dir)[0]
    #path_checkpoint = os.path.join(checkpoint_dir, 'model.ckpt')
    path_checkpoint = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
    model_lit = LitModelSurvSurf.load_from_checkpoint(path_checkpoint)
    
    datamodule = DataModuleNCT00981058(
        df_dir=ds_df_dir, 
        ds_name=ds_name, 
        g_resol=g_resol, 
        t_resol=t_resol,
        separate_g_from_feats=True, 
        train_mode='first_cross_obs_only',
        eval_mode='true_probs_grid',
        batch_size=100, 
        num_workers=4)
    datamodule.setup(stage=split)
    if 'val' in split:
        dataloader_obs, dataloader_true_prob = datamodule.val_dataloader()
    if 'test' in split:
        dataloader_obs, dataloader_true_prob = datamodule.test_dataloader()

    with torch.no_grad():
        torch.cuda.empty_cache()
    gc.collect()

    model_loaded_core = model_lit.model
    model_loaded_core = model_loaded_core.to(device)
    model_loaded_core.eval()


    out_dfs = dict()
    for key, dataloader in [
        ('obs', dataloader_obs),
        ('true_prob', dataloader_true_prob)
    ]:
        
        subjs = []
        g = []
        t = []
        pred = []
        truth = []
        with torch.no_grad():
            for batch_, (subj, xs, gs, ts, ys, weight, _) in enumerate(dataloader):
                xs, gs, ts, ys = xs.to(device), gs.to(device), ts.to(device), ys.to(device)
                out = model_loaded_core(ts, gs, xs)
                subjs += list(subj)
                g += list(datamodule.g_max*gs.cpu().numpy()[:,0])
                t += list(ts.cpu().numpy()[:,0])
                truth += list(ys.cpu().numpy()[:,0])
                pred += list(out.cpu().numpy()[:,0])


        pred = np.array(pred)
        truth = np.array(truth)
        df_val_results = pd.DataFrame()
        df_val_results['subj'] = subjs
        df_val_results['g'] = g
        df_val_results['t'] = t
        df_val_results['pred'] = pred
        df_val_results['truth'] = truth
        for col in ['pred', 'truth']:
            df_val_results[col] = df_val_results[col].astype(float)
        out_dfs[key] = df_val_results
    return out_dfs, model_loaded_core.__class__.__name__


In [68]:


ds_train = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='train', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
ds_val = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='val', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)
ds_test = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='test', 
    mode='true_probs_grid', 
    separate_g_from_feats=False
)

In [69]:
ds_test._get_df_Xy_trans_obs()['g_max_by_time'].value_counts()

g_max_by_time
2.0    66
1.0    65
3.0    64
4.0    45
5.0    17
6.0     2
Name: count, dtype: int64

In [70]:
max_time = 140

In [71]:
model_to_metric = []
model_to_prop_t_violated_per_subj = []
model_to_max_violated_per_subj = []
for run_id, pred_fun, descriptor in [
    ('vtcx5w3i',pred_from_SurvSurf,'LossDyDg'),
]:
    out_model, model_name = pred_fun(run_id=run_id, proj_name=proj_name, split=split)

/home/yc366/miniconda3/envs/env_survsurf_benchmark/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


In [72]:
from model_factory_survsurf import LitModelSurvSurf
checkpoint_dir = f'/home/yc366/repos/survsurf_benchmark/runtime_results/{proj_name}/{run_id}/checkpoints/'
path_checkpoint = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
model_lit = LitModelSurvSurf.load_from_checkpoint(path_checkpoint)

with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

model_loaded_core = model_lit.model

/home/yc366/miniconda3/envs/env_survsurf_benchmark/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


In [73]:
df_test = ds_test._get_df_Xy_true_prob()

In [74]:
df_test[['subject','feat__traj_clust']].drop_duplicates().sort_values('feat__traj_clust')

,subject,feat__traj_clust
0,b'014-7882',0.0
11760,b'668-7861',0.0
6240,b'192-7808',0.0
7080,b'413-7881',0.0
3480,b'146-7801',0.0
...,...,...
4200,b'146-7857',1.0
6000,b'162-7886',1.0
5760,b'161-7889',1.0
1200,b'044-7880',1.0


In [150]:
sbj1 = "b'014-7882'" #gray
sbj2 = "b'162-7886'" #green

In [151]:
g = 2
selector1 = (df_test['subject']==sbj1) & (df_test['g_max_by_time']==g)
x1 = df_test.loc[
    selector1,
    [i for i in df_test.columns if i.startswith('feat')]
].values
g1 = df_test.loc[
    selector1,
    ['g_max_by_time']
].values/ds.g_max
t1 = df_test.loc[
    selector1,
    ['duration']
].values

selector2 = (df_test['subject']==sbj2) & (df_test['g_max_by_time']==g)
x2 = df_test.loc[
    selector2,
    [i for i in df_test.columns if i.startswith('feat')]
].values
g2 = df_test.loc[
    selector2,
    ['g_max_by_time']
].values/ds.g_max
t2 = df_test.loc[
    selector2,
    ['duration']
].values

x1 = torch.tensor(x1, dtype=torch.float32).to(device)
x2 = torch.tensor(x2, dtype=torch.float32).to(device)
g1 = torch.tensor(g1, dtype=torch.float32).to(device)
g2 = torch.tensor(g2, dtype=torch.float32).to(device)
t1 = torch.tensor(t1, dtype=torch.float32, requires_grad=True).to(device)
t2 = torch.tensor(t2, dtype=torch.float32, requires_grad=True).to(device)


In [152]:
out1 = model_loaded_core(t1, g1, x1)
out2 = model_loaded_core(t2, g2, x2)

In [153]:
out1

tensor([[0.0000],
        [0.4555],
        [0.7553],
        [0.9002],
        [0.9603],
        [0.9839],
        [0.9932],
        [0.9970],
        [0.9986],
        [0.9993],
        [0.9996],
        [0.9998],
        [0.9999],
        [0.9999],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000]], grad_fn=<TanhBackward0>)

In [154]:
out2

tensor([[0.0000],
        [0.0622],
        [0.1247],
        [0.1868],
        [0.2475],
        [0.3059],
        [0.3615],
        [0.4136],
        [0.4620],
        [0.5064],
        [0.5467],
        [0.5832],
        [0.6159],
        [0.6451],
        [0.6710],
        [0.6939],
        [0.7142],
        [0.7320],
        [0.7478],
        [0.7616],
        [0.7738],
        [0.7845],
        [0.7940],
        [0.8024]], grad_fn=<TanhBackward0>)

In [155]:
alpha1 = []
for i in range(t1.shape[0]):
    t = t1[i:i+1,:]
    g = g1[i:i+1,:]
    x = x1[i:i+1,:]
    out = model_loaded_core(t, g, x)
    A = -torch.log(1-out)

    alpha1.append(torch.autograd.grad(A, t)[0][0][0].cpu().numpy())
alpha1 = np.array(alpha1)

In [156]:
alpha2 = []
for i in range(t2.shape[0]):
    t = t2[i:i+1,:]
    g = g2[i:i+1,:]
    x = x2[i:i+1,:]
    out = model_loaded_core(t, g, x)
    A = -torch.log(1-out)

    alpha2.append(torch.autograd.grad(A, t)[0][0][0].cpu().numpy())
alpha2 = np.array(alpha2)

In [157]:
alpha2/alpha1

array([0.12632386, 0.09275708, 0.08264422, 0.08240337, 0.08652467,
       0.09282171, 0.10041647, 0.10892662, 0.11814839, 0.12793756,
       0.13816676, 0.14871192, 0.15945008, 0.17026217, 0.18103527,
       0.19166635, 0.20206423, 0.21215151, 0.22186437, 0.23115343,
       0.23998266, 0.24832806, 0.256177  , 0.2635258 ], dtype=float32)

# Clean up

In [35]:
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

2162